# Part 4: Multivariate analysis

- **Data:** `data/listings_clean.csv` from `01_cleaning.ipynb`.
- **Main question:** which listing and host features predict a top rating (4.8 or higher), so hosts know what to improve first?
- **Target variable:** `top_rated` (1 = overall rating of 4.8 or higher, 0 = below 4.8), defined in `01_cleaning.ipynb`. Every chart below is about this target: Figure 18 shows the top-rated rate by borough, Figure 19 shows which features the model needs to predict it, and Figure 20 shows how the predicted top-rated chance changes with each lever.
- **Goal:** the univariate and bivariate parts looked at one variable at a time. Here the variables are combined, to check which single-variable patterns survive once other features are held fixed, and which features add predictive power on their own.
- **Samples:** general views use all listings. Anything about `top_rated` uses rated listings only. Results describe listings that already have reviews, not new listings.
- **Carried over from earlier decisions:** review sub-scores and `host_is_superhost` are never used as features (leakage). Host response columns are empty. `price` is not imputed; its missingness is kept as a flag. Classes are close to balanced, so no resampling.
- **Format:** three charts, each tied to the main question. Figure 18 is an interactive Plotly map with a **Play** button (saved as HTML in `figures/`). Figures 19 and 20 are static, in the same style as the earlier parts (saved as PNG).

In [ ]:
import os
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from IPython.display import Markdown, display
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

try:
    import plotly.graph_objects as go
except ImportError as e:
    raise ImportError("The interactive map needs Plotly: run  pip install -U plotly  and restart the kernel") from e

warnings.filterwarnings("ignore", category=FutureWarning)

# Same locations as the earlier notebooks
DATA_PATH = "../data/listings_clean.csv"
FIG_DIR = "../figures"
os.makedirs(FIG_DIR, exist_ok=True)

BLUE, GREY, RED, ORANGE, TEAL, PURPLE = "#2471A3", "#95A5A6", "#C0392B", "#E67E22", "#16A085", "#7D3C98"
RNG = 42


def clean_axes(ax):
    for s in ["top", "right"]:
        ax.spines[s].set_visible(False)


def save(fig, name):
    # Static charts: save as PNG like the earlier parts, then show
    fig.savefig(os.path.join(FIG_DIR, name + ".png"), dpi=500, bbox_inches="tight")
    plt.show()


def finish(fig, name, title, subtitle=None, height=520, bottom=None):
    # Interactive charts: shared look, saved as HTML in the figures folder, then shown.
    # `bottom` is the space (px) under the plot, needed for Play buttons and sliders.
    text = f"<b>{title}</b>" + (f"<br><span style='font-size:13px;color:#555555'>{subtitle}</span>" if subtitle else "")
    has_frames = bool(getattr(fig, "frames", None))
    fig.update_layout(template="plotly_white", font=dict(family="Arial, sans-serif", size=13),
                      margin=dict(l=60, r=30, t=95, b=bottom if bottom else (120 if has_frames else 60)),
                      title=dict(text=text, x=0.01, xanchor="left"), height=height)
    try:
        fig.write_html(os.path.join(FIG_DIR, name + ".html"), include_plotlyjs="cdn")
    except Exception as err:
        print("could not save", name, "->", err)
    fig.show()


def add_play(fig, ms=1500, y=-0.04):
    # Play / Pause buttons plus a slider for a figure whose frames were built by hand.
    # redraw=True is needed for map traces; transition=0 stops bubbles from sliding between frames.
    names = [f.name for f in fig.frames]
    fig.update_layout(
        updatemenus=[dict(type="buttons", direction="left", x=0.0, y=y, xanchor="left", yanchor="top", showactive=False,
                          buttons=[dict(label="Play", method="animate",
                                        args=[None, dict(frame=dict(duration=ms, redraw=True), fromcurrent=True,
                                                         transition=dict(duration=0), mode="immediate")]),
                                   dict(label="Pause", method="animate",
                                        args=[[None], dict(frame=dict(duration=0, redraw=False), mode="immediate",
                                                           transition=dict(duration=0))])])],
        sliders=[dict(active=0, x=0.16, len=0.84, y=y, yanchor="top", currentvalue=dict(prefix="Showing: "),
                      steps=[dict(label=n, method="animate",
                                  args=[[n], dict(mode="immediate", frame=dict(duration=0, redraw=True),
                                                  transition=dict(duration=0))]) for n in names])])


if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(f"{DATA_PATH} not found. Run 01_cleaning.ipynb first; it creates this file.")

df = pd.read_csv(DATA_PATH, low_memory=False)
rated = df[df["top_rated"].notna()].copy()
rated["top_rated"] = rated["top_rated"].astype(int)
BASE = rated["top_rated"].mean() * 100
print(df.shape, "| rated:", len(rated), f"| overall top-rated rate: {BASE:.1f}%")

## Step 1: Build the multivariate features

The earlier parts ended with a list of candidates and a list of cautions. This step turns them into model-ready columns, without touching the cleaned file.

- **Grouped categories:** `property_type` has a long tail, shared and hotel rooms are tiny, and capacity is grouped as in the bivariate part.
- **Host type:** `calculated_host_listings_count` grouped into 1, 2-5 and 6+ listings.
- **Amenities:** parsed from JSON text into a count and a set of mid-frequency amenities. Near-universal amenities cannot separate listings, so they are left out.
- **Price:** the log of `price_capped`, a `price_missing` flag (missing price was linked to listings that cannot be booked), and a new **relative price**: a listing's price divided by the median of similar listings (same borough, room type and capacity group). This shows whether a listing is cheap or expensive for what it offers, which is something a host can change.
- **Skewed counts:** log scale for reviews and minimum nights, as the univariate part recommended.

In [ ]:
CAP_BINS, CAP_LABELS = [0, 2, 4, 6, np.inf], ["1-2 guests", "3-4 guests", "5-6 guests", "7+ guests"]
HOST_ORDER = ["1 listing", "2-5 listings", "6+ listings"]
ROOM_ORDER = ["Entire home/apt", "Private room", "Shared or hotel room"]


def add_features(d):
    d = d.copy()
    d["room_group"] = d["room_type"].where(d["room_type"].isin(["Entire home/apt", "Private room"]), "Shared or hotel room")
    top_props = d["property_type"].value_counts().head(5).index
    d["property_group"] = d["property_type"].where(d["property_type"].isin(top_props), "Other property type")
    d["capacity_group"] = pd.cut(d["accommodates"], bins=CAP_BINS, labels=CAP_LABELS).astype(object)
    d["host_type"] = pd.cut(d["calculated_host_listings_count"], bins=[0, 1, 5, np.inf], labels=HOST_ORDER).astype(object)
    # price: log, missing flag, and price relative to similar listings
    d["price_missing"] = d["price_capped"].isna().astype(int)
    d["log_price"] = np.log(d["price_capped"])
    keys = ["neighbourhood_cleansed", "room_group", "capacity_group"]
    peer = d.groupby(keys)["price_capped"].transform("median")
    peer_n = d.groupby(keys)["price_capped"].transform("count")
    d["price_vs_peers"] = d["price_capped"] / peer.where(peer_n >= 10)  # needs at least 10 priced peers
    # skewed counts
    d["log_reviews"] = np.log1p(d["number_of_reviews"])
    d["log_min_nights"] = np.log1p(d["minimum_nights"].clip(upper=365))
    # amenities
    am = d["amenities"].apply(lambda s: json.loads(s) if isinstance(s, str) else [])
    d["amenity_count"] = am.str.len()
    d["_amenities"] = am
    return d


df = add_features(df)
rated = df[df["top_rated"].notna()].copy()
rated["top_rated"] = rated["top_rated"].astype(int)

# mid-frequency amenities: offered by 10-80% of rated listings (basics and brand names cannot separate listings)
amen_sets = rated["_amenities"].apply(lambda L: {x.strip().lower() for x in L})
am_share = amen_sets.explode().value_counts() / len(rated)
am_share = am_share[(am_share >= 0.10) & (am_share <= 0.80)]
AMEN = [a for a in am_share.sort_values(ascending=False).head(60).index if len(a) <= 40][:40]
amen_matrix = pd.DataFrame({a: amen_sets.apply(lambda s, a=a: a in s) for a in AMEN}, index=rated.index)

print("rated listings:", len(rated), "| with a relative price:", f"{int(rated['price_vs_peers'].notna().sum()):,}",
      "| mid-frequency amenities kept:", len(AMEN))
rated[["room_group", "capacity_group", "host_type", "price_vs_peers", "amenity_count"]].head()

## Figure 18: Where are top-rated listings, and does it depend on who the host is? (interactive map)

The univariate part left point maps for the dashboard because borough already carries the location. With several variables in play, a map shows what single-variable charts could not: whether high and low top-rated areas cluster together, and whether that changes with host type.

Each bubble is a borough. Its **size** is the number of rated listings and its **colour** is the top-rated rate (blue above the overall rate, red below). Hover for the median price. **Press Play** to step from all hosts to single-listing hosts, hosts with 2-5 listings, and hosts with 6 or more. Bubble size and colour use the same scale in every step, so the frames can be compared directly. You can also drag the slider underneath. Boroughs with fewer than 30 rated listings in a step are not shown.

*If Play does nothing in your editor, drag the slider instead, or open `figures/fig18_borough_map_by_host_type.html` in a browser. The map background needs an internet connection.*

In [ ]:
CENTER = dict(lat=float(df["latitude"].median()), lon=float(df["longitude"].median()))
RANGE = [max(0, BASE - 20), min(100, BASE + 20)]
FRAME_ORDER = ["All hosts"] + HOST_ORDER          # fixed order: Play starts at All hosts

cent = df.groupby("neighbourhood_cleansed")[["latitude", "longitude"]].mean()
parts = []
for label, d in [("All hosts", rated)] + [(h, rated[rated["host_type"] == h]) for h in HOST_ORDER]:
    s = d.groupby("neighbourhood_cleansed").agg(n=("top_rated", "size"), rate=("top_rated", "mean"),
                                                median_price=("price_capped", "median")).reset_index()
    s["rate"] = s["rate"] * 100
    s["Host type"] = label
    parts.append(s[s["n"] >= 30])
bm = pd.concat(parts).merge(cent.reset_index(), on="neighbourhood_cleansed")
bm["median_price"] = bm["median_price"].round(0)

# Plotly 5.24+ has Scattermap (MapLibre); older versions have Scattermapbox. Both work without a token.
USE_NEW_MAP = hasattr(go, "Scattermap")
MapTrace = go.Scattermap if USE_NEW_MAP else go.Scattermapbox
SIZEREF = 2 * bm["n"].max() / (44 ** 2)          # one bubble scale for every frame (largest bubble = 44 px)


def map_trace(d):
    return MapTrace(
        lat=d["latitude"], lon=d["longitude"], mode="markers", text=d["neighbourhood_cleansed"],
        marker=dict(size=d["n"], sizemode="area", sizeref=SIZEREF, sizemin=5, opacity=0.85,
                    color=d["rate"], colorscale="RdBu", cmin=RANGE[0], cmax=RANGE[1],
                    showscale=True, colorbar=dict(title=dict(text="Top-rated %"))),
        customdata=np.stack([d["n"], d["rate"], d["median_price"]], axis=-1),
        hovertemplate=("<b>%{text}</b><br>Rated listings: %{customdata[0]:,}<br>Top-rated: %{customdata[1]:.1f}%"
                       "<br>Median price: £%{customdata[2]:,.0f}<extra></extra>"),
        showlegend=False)


frames = [go.Frame(name=lab, data=[map_trace(bm[bm["Host type"] == lab])]) for lab in FRAME_ORDER]
fig = go.Figure(data=[map_trace(bm[bm["Host type"] == FRAME_ORDER[0]])], frames=frames)
map_layout = dict(style="open-street-map", center=CENTER, zoom=8.6)
fig.update_layout(**({"map": map_layout} if USE_NEW_MAP else {"mapbox": map_layout}))
add_play(fig, ms=1500, y=-0.04)
finish(fig, "fig18_borough_map_by_host_type", "Top-rated rate across London boroughs",
       "Bubble size = rated listings, colour = top-rated rate. Press Play or use the slider to change host type.",
       height=700, bottom=150)

allh = bm[bm["Host type"] == "All hosts"]
top3 = allh.nlargest(3, "rate")["neighbourhood_cleansed"].tolist()
bot3 = allh.nsmallest(3, "rate")["neighbourhood_cleansed"].tolist()
mv = bm.pivot_table(index="neighbourhood_cleansed", columns="Host type", values="rate")
shift = (mv["6+ listings"] - mv["1 listing"]).dropna() if {"6+ listings", "1 listing"} <= set(mv.columns) else pd.Series(dtype=float)
display(Markdown(
    f"**Insight:** the highest top-rated rates are in {', '.join(top3)} and the lowest in {', '.join(bot3)}. "
    + (f"Of the {len(shift)} boroughs with enough listings of both host types, hosts with 6+ listings do better than single-listing hosts in "
       f"{int((shift > 0).sum())} and worse in {int((shift < 0).sum())}, so the host-type gap is "
       f"{'not the same everywhere' if 0 < (shift > 0).sum() < len(shift) else 'in the same direction everywhere'}." if len(shift) else "")))

## Step 2: Model the target with all allowed features

The map shows one slice at a time. A gradient-boosted classifier uses all allowed features together, and can pick up non-linear effects and interactions that the bivariate part suggested (capacity and host tenure were not linear). Two decisions from earlier parts shape it:

- **No leakage:** review sub-scores and `host_is_superhost` are excluded. Latitude and longitude are excluded because borough already covers location.
- **Missing values:** the model handles missing values natively, so missing price, bedrooms and beds are not imputed. A `price_missing` flag is added because missingness was linked to listings that cannot be booked.

Features are put into groups, and the groups are added one at a time so the contribution of each is visible. Groups made of things a host can change are marked as **levers**.

In [ ]:
cat_cols = ["room_group", "property_group", "neighbourhood_cleansed"]
X = pd.DataFrame(index=rated.index)
for c in cat_cols:
    cd, _ = pd.factorize(rated[c])
    X[c] = np.where(cd < 0, np.nan, cd).astype(float)
for c in ["accommodates", "bedrooms", "beds", "bathrooms", "log_price", "price_missing", "price_vs_peers", "log_min_nights",
          "availability_365", "amenity_count", "log_reviews", "hosts_time_as_host_years"]:
    X[c] = rated[c].astype(float)
X["log_host_listings"] = np.log1p(rated["calculated_host_listings_count"].astype(float))
X["host_identity_verified"] = rated["host_identity_verified"].map({"t": 1.0, "f": 0.0})
HAS_INSTANT = "instant_bookable" in rated.columns
if HAS_INSTANT:
    X["instant_bookable"] = rated["instant_bookable"].map({"t": 1.0, "f": 0.0})
amen_cols = []
for a in AMEN:
    X["amenity: " + a] = amen_matrix[a].astype(float)
    amen_cols.append("amenity: " + a)

GROUPS = {
    "Fixed context": ["room_group", "property_group", "accommodates", "bedrooms", "beds", "bathrooms", "neighbourhood_cleansed"],
    "Host profile": ["hosts_time_as_host_years", "log_host_listings", "host_identity_verified"],
    "Pricing": ["log_price", "price_missing", "price_vs_peers"],
    "Booking rules": ["log_min_nights", "availability_365"] + (["instant_bookable"] if HAS_INSTANT else []),
    "Amenities": ["amenity_count"] + amen_cols,
    "Review activity": ["log_reviews"],
}
LEVERS = {"Pricing", "Booking rules", "Amenities"}
GROUP_COLORS = {g: (TEAL if g in LEVERS else GREY) for g in GROUPS}

y_all = rated["top_rated"].to_numpy()
X_tr, X_te, y_tr, y_te = train_test_split(X, y_all, test_size=0.25, stratify=y_all, random_state=RNG)


def fit(cols):
    mask = [c in cat_cols for c in cols]
    m = HistGradientBoostingClassifier(max_iter=250, learning_rate=0.07, max_leaf_nodes=31, min_samples_leaf=40,
                                       l2_regularization=1.0, early_stopping=True, validation_fraction=0.12, n_iter_no_change=15,
                                       categorical_features=mask if any(mask) else None, random_state=RNG)
    return m.fit(X_tr[cols], y_tr)


steps, cols_so_far, models = [], [], {}
for g, cols in GROUPS.items():
    cols_so_far = cols_so_far + cols
    m = fit(cols_so_far)
    steps.append(dict(group=g, auc=roc_auc_score(y_te, m.predict_proba(X_te[cols_so_far])[:, 1])))
    models[g] = (m, list(cols_so_far))
model, MODEL_COLS = models["Review activity"]
print("train:", len(X_tr), "| test:", len(X_te))
names = [s["group"] for s in steps]
aucs = [s["auc"] for s in steps]
gains = np.diff([0.5] + aucs)
print("final model AUC:", round(aucs[-1], 3), "(0.5 = coin toss, 1.0 = perfect)")
pd.DataFrame({"group added": names, "test AUC": np.round(aucs, 3), "AUC gained": np.round(gains, 3)})

## Figure 19: Which features matter most for being top rated?

**In plain words:** we ask a computer model to guess, for every listing, whether it is top rated. It uses everything we know about the listing: room type, size, borough, price, amenities, booking rules and host details. Then each feature is tested: its values are shuffled at random, which breaks its link to the rating, and we check how much worse the guesses get. **A big drop means the feature matters a lot. A drop near zero means it adds little.**

**How to read the chart**
- Left panel: features grouped together (for example, all the price features count as one group).
- Right panel: the 15 single features that matter most.
- Teal bars are things a host can change (price, amenities, booking rules). Grey bars are things a host cannot easily change (location, room type, size, host history, number of reviews).
- The thin line on each bar shows how much the result moves between repeats; a bar whose line crosses zero is not clearly different from nothing.

In [ ]:
test_sub = X_te.sample(min(12000, len(X_te)), random_state=RNG)
y_sub = pd.Series(y_te, index=X_te.index).loc[test_sub.index].to_numpy()
base_auc = roc_auc_score(y_sub, model.predict_proba(test_sub[MODEL_COLS])[:, 1])


def perm_drop(cols, repeats=5, seed=RNG):
    rng = np.random.default_rng(seed)
    drops = []
    for _ in range(repeats):
        Xp = test_sub[MODEL_COLS].copy()
        Xp[cols] = test_sub[cols].to_numpy()[rng.permutation(len(test_sub))]
        drops.append(base_auc - roc_auc_score(y_sub, model.predict_proba(Xp)[:, 1]))
    return float(np.mean(drops)), float(np.std(drops))


gi = pd.DataFrame([(g, *perm_drop(c)) for g, c in GROUPS.items()], columns=["group", "drop", "sd"]).sort_values("drop")
fi = pd.DataFrame([(c, *perm_drop([c], repeats=3)) for c in MODEL_COLS], columns=["feature", "drop", "sd"])
feat_group = {c: g for g, cs in GROUPS.items() for c in cs}
fi["group"] = fi["feature"].map(feat_group)
fi = fi.sort_values("drop").tail(15)

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5), gridspec_kw={"width_ratios": [1, 1.3]}, layout="constrained")
def value_labels(ax, d, fs):
    # write each value just past the right end of its whisker, so it never collides with the axis labels
    pad = (d["drop"] + d["sd"]).max() * 0.02
    for yi, (v, s) in enumerate(zip(d["drop"], d["sd"])):
        ax.text(max(v + s, 0) + pad, yi, f"{v:.3f}", va="center", fontsize=fs)
    ax.set_xlim(min(0, (d["drop"] - d["sd"]).min()) * 1.15, (d["drop"] + d["sd"]).max() * 1.2)


ax = axes[0]
ax.barh(gi["group"], gi["drop"], xerr=gi["sd"], color=[GROUP_COLORS[g] for g in gi["group"]], error_kw=dict(lw=1, capsize=2))
value_labels(ax, gi.reset_index(drop=True), 8)
ax.tick_params(axis="y", labelsize=9)
ax.axvline(0, color="#444444", lw=0.8)
ax.set_xlabel("How much worse the model predicts a top rating when this is shuffled")
ax.set_title("Groups of features", loc="left", fontsize=9)
clean_axes(ax)
ax = axes[1]
ax.barh(fi["feature"].str.replace("_", " "), fi["drop"], xerr=fi["sd"], color=[GROUP_COLORS[g] for g in fi["group"]],
        error_kw=dict(lw=1, capsize=2))
value_labels(ax, fi.reset_index(drop=True), 7)
ax.axvline(0, color="#444444", lw=0.8)
ax.tick_params(axis="y", labelsize=9)
ax.set_xlabel("How much worse the model predicts a top rating when this is shuffled")
ax.set_title("The 15 most important single features", loc="left", fontsize=9)
clean_axes(ax)
ax.legend(handles=[Patch(facecolor=TEAL, label="A host can change this"), Patch(facecolor=GREY, label="A host cannot easily change this")],
          loc="lower right", fontsize=8, frameon=False)

lever_rank = gi[gi["group"].isin(LEVERS)].sort_values("drop", ascending=False)
top_feat = fi.sort_values("drop", ascending=False).iloc[0]
fig.suptitle(f"{gi.iloc[-1]['group']} matters most overall; among things a host can change, {lever_rank.iloc[0]['group'].lower()} matters most",
             x=0.01, ha="left", fontsize=11)
save(fig, "fig19_permutation_importance")

display(Markdown(
    f"**Insight:** using all allowed features, the model separates top-rated from other listings with a score of {aucs[-1]:.3f} "
    f"(0.5 = no better than guessing, 1.0 = perfect); fixed context alone scores {aucs[0]:.3f}. "
    f"Shuffling **{gi.iloc[-1]['group']}** hurts the model most (drop {gi.iloc[-1]['drop']:.3f}). Among the things a host can change, the order is "
    f"{' > '.join(f'**{g}** ({d:.3f})' for g, d in zip(lever_rank['group'], lever_rank['drop']))}. "
    f"The single most important feature is `{top_feat['feature']}` ({top_feat['drop']:.3f}). "
    f"This shows what the model uses to predict a top rating, not what causes one. Figure 20 shows which direction each feature pushes."))


## Figure 20: If a host changes this, does the chance of a top rating go up or down?

**In plain words:** each small chart takes one feature. Along the bottom is the value of that feature (for example 10, 20 or 30 amenities). Up the side is the model's predicted chance that a listing is top rated if it had that value, with everything else about the listing left as it is.

**How to read the chart**
- A line that **rises** means more of this feature goes with a higher chance of a top rating. A line that **falls** means the opposite. A **flat** line means it makes little difference.
- Teal lines are things a host can change. Grey lines are shown for comparison only (a host cannot set the number of reviews or years of experience directly).
- The shaded band is where most listings sit (the middle 80%). Trust the line most inside the band.
- The dashed line is the average chance for all listings. All six charts use the same vertical scale, so you can compare how much each one matters.
- This shows a pattern in the data, not proof that changing a feature will raise a rating.

In [ ]:
pdp_base = X_te.sample(min(3000, len(X_te)), random_state=RNG)[MODEL_COLS]
REF = model.predict_proba(pdp_base)[:, 1].mean() * 100
amen_grid = np.unique(np.percentile(rated["amenity_count"], [2, 10, 25, 40, 50, 60, 75, 90, 98]).round())
ten_max = int(np.nanmax(rated["hosts_time_as_host_years"]))
# (label, model column, grid in real units, transform to model scale, real column for the shaded band, x label, x scale, lever?)
PDP_SPECS = [
    ("Price vs similar listings", "price_vs_peers", np.array([0.5, 0.7, 0.85, 1.0, 1.15, 1.3, 1.6, 2.0]), None,
     "price_vs_peers", "Price as a multiple of similar listings (1.0 = typical)", "linear", True),
    ("Amenities listed", "amenity_count", amen_grid, None, "amenity_count", "Number of amenities listed", "linear", True),
    ("Minimum nights", "log_min_nights", np.array([1, 2, 3, 4, 5, 7, 14, 30]), np.log1p, "minimum_nights",
     "Minimum nights required (log scale)", "log", True),
    ("Days available (next 365)", "availability_365", np.array([0, 30, 90, 150, 210, 270, 330, 365]), None, "availability_365",
     "Days available in the next 365", "linear", True),
    ("Reviews so far (context)", "log_reviews", np.array([0, 1, 2, 5, 10, 25, 50, 100, 200]), np.log1p, "number_of_reviews",
     "Number of reviews (log-like scale)", "symlog", False),
    ("Host tenure (context)", "hosts_time_as_host_years", np.unique(np.clip([0, 1, 2, 3, 5, 7, 9, 12], 0, ten_max)), None,
     "hosts_time_as_host_years", "Years as a host", "linear", False),
]


def pdp_curve(col, grid, tf):
    base = pdp_base[pdp_base["price_vs_peers"].notna()] if col == "price_vs_peers" else pdp_base
    out = []
    for v in grid:
        Xs = base.copy()
        Xs[col] = tf(v) if tf else v
        out.append(model.predict_proba(Xs)[:, 1].mean() * 100)
    return np.array(out)


curves = [(*spec, pdp_curve(spec[1], spec[2], spec[3])) for spec in PDP_SPECS]
lo_y = min(c[-1].min() for c in curves) - 1.5
hi_y = max(c[-1].max() for c in curves) + 1.5

fig, axes = plt.subplots(2, 3, figsize=(14, 7.2), sharey=True, layout="constrained")
for ax, (lab, col, grid, tf, orig, xt, xs, is_lever, vals) in zip(axes.ravel(), curves):
    p10, p90 = rated[orig].quantile([0.1, 0.9])
    ax.axvspan(p10, p90, color="#EAF2F8", zorder=0)
    ax.axhline(REF, color="#444444", ls="--", lw=1)
    ax.plot(grid, vals, color=TEAL if is_lever else GREY, marker="o", lw=2.5, ms=6, zorder=3)
    if xs == "log":
        ax.set_xscale("log")
        ax.set_xticks([1, 2, 5, 10, 30])
        ax.set_xticklabels(["1", "2", "5", "10", "30"])
    elif xs == "symlog":
        ax.set_xscale("symlog", linthresh=1)
        ax.set_xticks([0, 1, 5, 25, 100, 200])
        ax.set_xticklabels(["0", "1", "5", "25", "100", "200"])
    ax.set_xlabel(xt, fontsize=8)
    ax.set_title(lab, loc="left", fontsize=9)
    ax.set_ylim(lo_y, hi_y)
    clean_axes(ax)
for ax in axes[:, 0]:
    ax.set_ylabel("Predicted chance of being top rated (%)")
axes[0, 0].text(0.02, 0.04, f"Sample average {REF:.1f}%", transform=axes[0, 0].transAxes, fontsize=8, color="#444444")
fig.legend(handles=[Patch(facecolor=TEAL, label="A host can change this"), Patch(facecolor=GREY, label="Shown for comparison"),
                    Patch(facecolor="#EAF2F8", label="Middle 80% of listings")],
           loc="outside lower center", ncol=3, fontsize=8, frameon=False)

pdp_sum = pd.DataFrame([dict(lever=c[0], low=c[-1].min(), high=c[-1].max(), best=c[2][int(np.argmax(c[-1]))],
                             worst=c[2][int(np.argmin(c[-1]))], span=c[-1].max() - c[-1].min(), is_lever=c[7]) for c in curves]
                       ).sort_values("span", ascending=False)
top_move = pdp_sum[pdp_sum["is_lever"]].iloc[0]
fig.suptitle(f"Of the things a host can change, {top_move['lever'].lower()} makes the biggest difference: "
             f"predicted chance goes from {top_move['low']:.1f}% to {top_move['high']:.1f}%", x=0.01, ha="left", fontsize=11)
save(fig, "fig20_lever_response_curves")

lines = "; ".join(f"**{r.lever}**: {r.low:.1f}% to {r.high:.1f}% (highest at {r.best:g}, lowest at {r.worst:g})" for r in pdp_sum.itertuples())
display(Markdown(
    f"**Insight:** predicted chance of a top rating for each feature, biggest change first: {lines}. The average is {REF:.1f}%. "
    f"A line that barely moves means that feature makes little difference to the prediction. "
    f"These are patterns among listings that already have reviews, not proof that changing a feature will raise a rating."))


## Step 3: What to improve first

The levers are ranked by how much the model relies on them (Figure 19), and each is paired with what its curve showed (Figure 20). The table is filled from the results above.

In [ ]:
RESPONSE_OF = {"Pricing": ["Price vs similar listings"], "Amenities": ["Amenities listed"],
               "Booking rules": ["Minimum nights", "Days available (next 365)"]}
pdp_idx = pdp_sum.set_index("lever")
rows = []
for rank, (g, d) in enumerate(zip(lever_rank["group"], lever_rank["drop"]), start=1):
    resp = "; ".join(f"{k}: {pdp_idx.loc[k, 'low']:.0f}% to {pdp_idx.loc[k, 'high']:.0f}%, highest at {pdp_idx.loc[k, 'best']:g}"
                     for k in RESPONSE_OF[g] if k in pdp_idx.index)
    rows.append(f"| {rank} | **{g}** | {d:.3f} | {resp} |")
display(Markdown("| Priority | Lever | Importance (drop when shuffled) | What the model's curve shows |\n|---|---|---|---|\n" + "\n".join(rows)))
display(Markdown(
    f"Fixed context (room type, size, borough) is not a lever, but it matters for expectations: its importance is "
    f"{gi.set_index('group').loc['Fixed context', 'drop']:.3f}, so a host should compare results with similar listings, not with the whole city."))

## Step 4: Final decision table

Two tables. **Table A** lists every analysis choice in this part and why, in the same format as the earlier parts (cleaning choices are in the cleaning part). **Table B** is the final call on each feature group: whether it stays in the model and how it should be used, based on the importance results above.

In [ ]:
n_rated, n_all = len(rated), len(df)
n_small = int((rated["room_group"] == "Shared or hotel room").sum())
verified = (rated["host_identity_verified"] == "t").mean() * 100
display(Markdown(f"""
**Table A: analysis decisions**

| Column / issue | Decision | Because |
|---|---|---|
| Sample | {n_rated:,} rated listings for everything involving `top_rated`; all {n_all:,} only where noted | A missing target cannot be filled without inventing answers; results apply to reviewed listings only |
| Review sub-scores, `host_is_superhost` | Never used as features | Leakage: they come from the same stays as the rating or depend on it |
| Host response rate, acceptance rate, response time | Not used | Completely empty in this snapshot |
| `latitude`, `longitude` | Used for the map only; `neighbourhood_cleansed` is the location feature | Borough already covers location; the map shows how location and host type interact |
| `price` | `log(price_capped)` plus a `price_missing` flag; missing values left to the model | Skew; missing price is linked to listings that cannot be booked, so it carries signal |
| Relative price (new) | `price_capped` divided by the median of listings in the same borough, room type and capacity group (at least 10 priced peers) | Raw price mixes location and size; relative price is also something a host can change |
| `room_type` | Shared and hotel rooms merged into one group | Only {n_small:,} rated listings, too small to compare separately |
| `property_type` | Top 5 kept, the rest grouped as "Other property type" | Long tail of rare labels |
| `accommodates` | Numeric in the model; grouped (1-2, 3-4, 5-6, 7+) in charts | Not linear in the bivariate part; complete and the main size measure |
| `bedrooms`, `beds` | Kept with missing values, not imputed | Imputation is a modelling choice; the model handles missing values natively |
| `neighbourhood_cleansed` | All boroughs kept; groups under 30 rated listings hidden in charts | Rates from small groups are unstable (uneven borough sizes) |
| `calculated_host_listings_count` | Grouped 1, 2-5, 6+ in charts; `log1p` in the model | Long right tail |
| `hosts_time_as_host_years` | Numeric in the model and in the curves; no linear effect assumed | The bivariate part found a non-linear pattern |
| `host_identity_verified` | In the model | Kept after the bivariate part, but {verified:.1f}% of rated listings are verified, so its contribution is checked, not assumed |
| `host_has_profile_pic` | Not used | Near-universal, so it carries little information |
| `number_of_reviews` | `log1p`; own group "Review activity", not counted as a lever | Result of past stays, not something a host sets |
| `minimum_nights`, `availability_365` | `log1p` for minimum nights; availability as is | Skew; 0 days available is ambiguous (booked or blocked) |
| `amenities` | Amenity count plus up to {len(AMEN)} amenities offered by 10-80% of listings | Basics cannot separate listings; the count and rarer amenities were the candidates |
| Small groups | Hidden below 30 listings (20 per map cell); 95% intervals where shown | Small samples give unstable rates |
| Classes | No resampling | Classes are close to balanced ({BASE:.0f}% top rated) |
| Model | Gradient-boosted classifier, 75/25 stratified split, scored by AUC | Handles missing values, non-linear effects and interactions |
| Importance | Whole groups scrambled together, repeated 5 times | Related features share credit, so single features understate their group |
| Charts | Three charts only: an interactive Plotly map with Play (Fig. 18) and two static charts (Fig. 19, 20) | Each answers a part of the main question: where, what matters, and which way to move |
"""))

# Table B: final decision for each feature group, filled from the results
gdrop = gi.set_index("group")
step_gain = dict(zip(names, gains))
CONTENT = {g: ", ".join(c.replace("_", " ") for c in cols[:4]) + (f" and {len(cols) - 4} more" if len(cols) > 4 else "")
           for g, cols in GROUPS.items()}
USE = {"Fixed context": "Use it to pick comparison listings, so hosts judge themselves against similar ones",
       "Host profile": "Context only; a host cannot change tenure or portfolio size in the short run",
       "Pricing": "Advise on price compared with similar listings, not the whole city (Fig. 20)",
       "Booking rules": "Secondary advice unless its curve in Fig. 20 moves clearly",
       "Amenities": "Advise on the amenity count (Fig. 20); individual amenities are not ranked here",
       "Review activity": "Control only; it keeps other effects honest but a host cannot set it"}
rows = []
for g in GROUPS:
    d, sd = gdrop.loc[g, "drop"], gdrop.loc[g, "sd"]
    keep = d > max(2 * sd, 0.002)
    verdict = "**Keep in the final model**" if keep else "**Weak signal: re-test without it**"
    rows.append(f"| {g} | {'Lever' if g in LEVERS else 'Context'} | {CONTENT[g]} | {d:.3f} (±{sd:.3f}) | {step_gain[g]:+.3f} | {verdict}. {USE[g]} |")
display(Markdown("**Table B: final decision for each feature group**\n\n"
                 "| Feature group | Role | Main features | Importance (drop when shuffled) | Score gained when the group is added | Final decision |\n|---|---|---|---|---|---|\n"
                 + "\n".join(rows)
                 + "\n\n*Rule used: a group is kept if scrambling it lowers AUC by more than twice its spread and by at least 0.002; otherwise it is flagged as a weak signal and should be re-tested without it (for example with cross-validation) before the final model is fixed.*"))

## Step 5: Challenges

- **Association is not cause.** Guests rate against their own expectations, and cleanliness, host communication and the exact stay are not in the data. A lever that predicts top ratings may not raise a rating when changed.
- **Only reviewed listings.** New listings with no rating are excluded, so the findings may not hold for them.
- **Overlapping features.** Size variables, price measures and amenity counts move together, so single-feature importance is shared between them. Groups are the safer reading.
- **Amenities mark home types.** Many mid-frequency amenities describe a kind of property, so an amenity the model likes is not necessarily one a host can add to improve a rating.
- **Ratings sit near the top.** Median ratings are close to the 4.8 cut point (see the univariate part), so small differences decide the class, which limits how well any model can do.
- **Thin geography.** Outer boroughs have few listings, so single bubbles on the map are indicative only.

## Step 6: Key takeaways

In [ ]:
top3 = allh.nlargest(3, "rate")["neighbourhood_cleansed"].tolist()
bot3 = allh.nsmallest(3, "rate")["neighbourhood_cleansed"].tolist()
lever_text = ", ".join(f"{g} ({d:.3f})" for g, d in zip(lever_rank["group"], lever_rank["drop"]))
display(Markdown(f"""
- **Geography:** the top-rated rate is highest in {", ".join(top3)} and lowest in {", ".join(bot3)}, and the pattern shifts with host type (Figure 18).
- **Combined view:** the full model reaches AUC {aucs[-1]:.3f} on unseen rated listings; fixed context alone reaches {aucs[0]:.3f} (Figure 19).
- **Levers, by importance:** {lever_text} (Figure 19). The direction and size of each lever are in Figure 20.
- **Pricing is relative:** compare a listing with similar ones, not with the whole city.
- **Next:** check the model with cross-validation, test a model that uses levers only, and put the map and the lever ranking on the dashboard.
"""))